In [1]:
import sys
import os

sys.path.append(os.path.abspath("../Src"))

from log.utils.logger import setup_logger

logger.py location: e:\Practicum Project\Practicum\Src\log\utils\logger.py
Log directory: e:\Practicum Project\Practicum\Src\log\Log_Data


In [3]:
logger = setup_logger("11_GridSearch")
logger.info("Starting normalization pipeline...")

2026-06-24 22:50:39 | INFO | Starting normalization pipeline...


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False

In [5]:
X_train = pd.read_csv("../Data/Processed/X_train.csv")
X_test = pd.read_csv("../Data/Processed/X_test.csv")

y_train = pd.read_csv("../Data/Processed/y_train.csv").squeeze()
y_test = pd.read_csv("../Data/Processed/y_test.csv").squeeze()

Fill Missing Values

In [6]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

Model Dictonary

In [7]:
models = {

    "Linear Regression": (
        LinearRegression(),
        {}
    ),

    "Ridge": (
        Ridge(),
        {
            "alpha":[0.1,1,10]
        }
    ),

    "Lasso": (
        Lasso(),
        {
            "alpha":[0.001,0.01,0.1]
        }
    ),

    "Decision Tree": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth":[5,10,None],
            "min_samples_split":[2,5]
        }
    ),

    "Random Forest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators":[100,200],
            "max_depth":[5,10,None]
        }
    ),

    "Gradient Boosting": (
        GradientBoostingRegressor(random_state=42),
        {
            "n_estimators":[100,200],
            "learning_rate":[0.01,0.1]
        }
    ),

    "Extra Trees": (
        ExtraTreesRegressor(random_state=42),
        {
            "n_estimators":[100,200]
        }
    ),

    "AdaBoost": (
        AdaBoostRegressor(random_state=42),
        {
            "n_estimators":[50,100]
        }
    ),

    "KNN": (
        KNeighborsRegressor(),
        {
            "n_neighbors":[3,5,7]
        }
    ),

    "SVR": (
        SVR(),
        {
            "C":[1,10],
            "kernel":["linear","rbf"]
        }
    ),

    "MLP": (
        MLPRegressor(random_state=42,max_iter=1000),
        {
            "hidden_layer_sizes":[(50,),(100,)],
            "alpha":[0.0001,0.001]
        }
    )
}

In [8]:
#optional  xgb boost
if xgb_available:

    models["XGBoost"] = (

        XGBRegressor(
            random_state=42,
            objective="reg:squarederror"
        ),

        {
            "n_estimators":[100,200],
            "max_depth":[3,5],
            "learning_rate":[0.05,0.1]
        }
    )

In [ ]:

logger.info("Initializing Grid Search Loop for model Selection...")
results = []

best_model = None
best_score = -999999
best_name = ""

2026-06-24 22:53:02 | INFO | Starting Grid Search Loop for model Selection...


In [ ]:
logger.info("Started Grid Search Loop for model Selection...")
for name, (model, params) in models.items():

    print("="*60)
    print(name)

    grid = GridSearchCV(

        estimator=model,

        param_grid=params,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    rmse = mean_squared_error(
        y_test,
        y_pred
    ) ** 0.5

    results.append({

        "Model":name,

        "Best CV Score":grid.best_score_,

        "Test R2":r2,

        "RMSE":rmse,

        "Best Params":grid.best_params_

    })

    if r2 > best_score:

        best_score = r2
        best_model = grid.best_estimator_
        best_name = name

    print("Best CV :",grid.best_score_)
    print("Test R2 :",r2)
    print("RMSE :",rmse)
    logger.info("Completed Grid Search Loop for model Selection...")

2026-06-24 22:53:57 | INFO | Started Grid Search Loop for model Selection...


Linear Regression
Best CV : -3.291492424774229
Test R2 : 0.11162824234134472
RMSE : 1.707003036060683
Ridge
Best CV : 0.007611640959171728
Test R2 : 0.1037140171235621
RMSE : 1.7145897537996417
Lasso
Best CV : 0.014361956202636717
Test R2 : 0.04824511496018613
RMSE : 1.7668491794521086
Decision Tree
Best CV : -0.5725623232457354
Test R2 : -0.15865263397781137
RMSE : 1.9494564984752085
Random Forest
Best CV : -0.05147527863305825
Test R2 : 0.074099341957371
RMSE : 1.7426859035350641
Gradient Boosting
Best CV : -0.05830064148165306
Test R2 : 0.00788057540278475
RMSE : 1.8039267481466275
Extra Trees
Best CV : -0.1895213215538139
Test R2 : -0.11235243902439018
RMSE : 1.910108897419202
AdaBoost
Best CV : -0.052288701600412615
Test R2 : 0.10428364320112715
RMSE : 1.7140448215552309
KNN
Best CV : -0.050829423127744786
Test R2 : 0.029492284718765638
RMSE : 1.7841707614806517
SVR
Best CV : -0.03206963511886873
Test R2 : 0.04708000647206667
RMSE : 1.7679303093650558
MLP
Best CV : -0.291926455167

In [11]:
logger.info("Results of Grid Search Loop for model Selection...")
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test R2",
    ascending=False
)

results_df

2026-06-24 22:55:45 | INFO | Results of Grid Search Loop for model Selection...


,Model,Best CV Score,Test R2,RMSE,Best Params
0,Linear Regression,-3.291492,0.111628,1.707003,{}
7,AdaBoost,-0.052289,0.104284,1.714045,{'n_estimators': 50}
1,Ridge,0.007612,0.103714,1.714590,{'alpha': 10}
4,Random Forest,-0.051475,0.074099,1.742686,"{'max_depth': 5, 'n_estimators': 200}"
2,Lasso,0.014362,0.048245,1.766849,{'alpha': 0.1}
9,SVR,-0.032070,0.047080,1.767930,"{'C': 1, 'kernel': 'rbf'}"
8,KNN,-0.050829,0.029492,1.784171,{'n_neighbors': 7}
5,Gradient Boosting,-0.058301,0.007881,1.803927,"{'learning_rate': 0.01, 'n_estimators': 100}"
10,MLP,-0.291926,-0.063305,1.867523,"{'alpha': 0.001, 'hidden_layer_sizes': (50,)}"
6,Extra Trees,-0.189521,-0.112352,1.910109,{'n_estimators': 200}


In [15]:
print("Best Model :",best_name)
print("Best Test R2 :",best_score)
logger.info(f"Best Model : {best_name}")
logger.info(f"Best Test R2 : {best_score}")

2026-06-24 22:58:44 | INFO | Best Model : Linear Regression
2026-06-24 22:58:44 | INFO | Best Test R2 : 0.11162824234134472


Best Model : Linear Regression
Best Test R2 : 0.11162824234134472


In [13]:
import joblib

joblib.dump(

    best_model,

    "../model/best_regression_model.pkl"

)

['../model/best_regression_model.pkl']